# Суу v2 — хоёр дахь сургалт (нэг загвар, 5 чадвар)

Эхний сургалт (Desmos хичээл, `sat_desmos_kit`) дууссаны **дараа** ажиллуулна. Тэр загвар дээрээ үргэлжлүүлж сургана, тиймээс Desmos-ын мэдлэгээ алдахгүй (хуучин хичээлүүдээс давтан оруулна).

| Чадвар | Апп-ын хэсэг | Юу сурах вэ |
|---|---|---|
| `chat` | AI багш Суу (/tutor) | hint, тайлбар, буруу хариуг "зөв" гэхгүй, англи үг, SAT-ын ерөнхий асуулт, латин үсгээр бичсэн асуулт |
| `stepcheck` | Суу чат, дасгал | сурагчийн бодолтоос **эхний буруу алхмыг** олох |
| `planner` | AI төлөвлөгөө (/planner) | сул сэдэв, хэв маяг олж, тэмдэглэлийг ("Мягмар гарагт завгүй") уншаад 7 хоногийн хуваарь гаргах (JSON) |
| `analyze` | Алдааны дэвтэр, бүтэн тестийн дүн | алдааны хэв маягийг олох, яг тоолох (JSON) |
| `desmos` | Desmos Академи (/desmos/ai) | эхний сургалтаас — алхам алхмаар Desmos хичээл (JSON) |
| `desmos` (panel) | Бодлогын хажуугийн муурны «Desmos» товч (дасгал, AI багш) | ижил хичээл, гэхдээ апп-ын асуултын хэлбэрээр: `A) …` сонголтууд, LaTeX (`\le`, `\frac{1}{2}`), зөв сонголтын утга эсвэл үсэг (`ai/v2/desmos-panel.ts`) |

**Colab дээр:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Зүүн талын Files хэсэгт 2 файл upload хийнэ: эхний сургалтын **`results.zip`** (дотор нь `sat-desmos-lora/`) ба **`sat_desmos_kit.zip`**.
3. Runtime → **Run all**. Хугацааг хэмжээгүй — 3B загвар дээр ойролцоогоор 1.5–2.5 цаг гэж тооцож байна. Хурдан туршихын тулд `FAST = True`.

Төгсгөлд `suu_v2.zip` (Ollama-д зориулсан GGUF + Modelfile + prompts.json + тайлан) ба `suu_v2_lora.zip` татагдана.

In [ ]:
%%capture
!pip install unsloth
!curl -fsSL https://bun.sh/install | bash

In [ ]:
# ---- Settings ----
REPO = "pinecone-studio/pinequest-s5-episode-2-team-12"
BRANCH = "feat/desmos-side-panel"   # change to "main" after the PR is merged
START = "desmos"           # "desmos" = continue from the first training (results.zip) · "base" = start from BASE_MODEL
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct"   # only used when START == "base"
FAST = False               # True = small quick run to check that everything works (~30–40 min)

EPOCHS = 1
LR = 1e-4                  # lower than a first training, so the Desmos skill is not overwritten
MAX_LEN = 2048
# training rows per skill (desmos = old lessons mixed back in so the model doesn't forget them)
CAP = {"chat": 2200, "planner": 700, "analyze": 700, "stepcheck": 600, "desmos": 500, "desmos_panel": 400}
# test rows per skill, scored before and after training
EVAL_N = {"chat": 30, "planner": 20, "analyze": 20, "stepcheck": 30, "desmos": 16, "desmos_panel": 16}
BEFORE_EVAL = True         # score the first model too (the "before" column for the slides)
BATCH = 8                  # prompts generated at once during the tests
if FAST:
    CAP = {k: v // 4 for k, v in CAP.items()}
    EVAL_N = {k: max(4, v // 3) for k, v in EVAL_N.items()}

BUN = "/root/.bun/bin/bun"
print("settings ok", CAP, EVAL_N)

In [ ]:
# ---- Get the v2 data and the scorer from the repo (small files only) ----
import os, json, subprocess
RAW = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}"
FILES = ["ai/data/tuning-v2/train.jsonl", "ai/data/tuning-v2/val.jsonl", "ai/data/tuning-v2/test.jsonl",
         "ai/data/tuning-v2/prompts.json", "ai/v2/skills.ts", "ai/v2/planner.ts", "ai/v2/analyze.ts",
         "ai/v2/stepcheck.ts", "ai/v2/score.ts", "ai/v2/desmos-panel.ts", "ai/pipeline/templates/shared.ts"]
os.makedirs("repo", exist_ok=True)
for f in FILES:
    os.makedirs(os.path.dirname(f"repo/{f}"), exist_ok=True)
    r = subprocess.run(["wget", "-q", "-O", f"repo/{f}", f"{RAW}/{f}"])
    if r.returncode != 0 or os.path.getsize(f"repo/{f}") == 0:
        raise SystemExit(f"Could not download {f}. Is BRANCH = '{BRANCH}' right?")
V2 = "repo/ai/data/tuning-v2"
print(subprocess.run([BUN, "--version"], capture_output=True, text=True).stdout.strip() or "bun missing!")

def read_jsonl(path):
    with open(path, encoding="utf-8") as fh:
        return [json.loads(l) for l in fh if l.strip()]

v2_train, v2_val, v2_test = (read_jsonl(f"{V2}/{n}.jsonl") for n in ("train", "val", "test"))
PROMPTS = json.load(open(f"{V2}/prompts.json", encoding="utf-8"))
from collections import Counter
print("train", Counter(r["skill"] for r in v2_train))
print("test ", Counter(r["skill"] for r in v2_test))

In [ ]:
# ---- Find the uploads: first-training adapter (results.zip) and the Desmos kit (sat_desmos_kit.zip) ----
import glob, zipfile, sys

def unzip_all():
    for z in glob.glob("/content/*.zip") + glob.glob("*.zip"):
        name = os.path.splitext(os.path.basename(z))[0]
        if name.startswith("suu_v2"):
            continue
        dest = f"uploads/{name}"
        if not os.path.isdir(dest):
            with zipfile.ZipFile(z) as zf:
                zf.extractall(dest)
            print("unzipped", z)

def find_adapter():
    hits = [p for p in glob.glob("uploads/**/adapter_config.json", recursive=True)]
    return os.path.dirname(sorted(hits, key=len)[0]) if hits else None

def find_kit():
    for p in glob.glob("uploads/**/data/train.jsonl", recursive=True):
        row = json.loads(open(p, encoding="utf-8").readline())
        if "technique" in row:
            return os.path.dirname(os.path.dirname(p))
    return None

unzip_all()
ADAPTER, KIT = find_adapter(), find_kit()
if (START == "desmos" and not ADAPTER) or not KIT:
    print("Файл олдсонгүй. results.zip ба sat_desmos_kit.zip-ээ сонгоно уу.")
    from google.colab import files
    files.upload()
    unzip_all()
    ADAPTER, KIT = find_adapter(), find_kit()
if START == "desmos" and not ADAPTER:
    raise SystemExit("results.zip дотор adapter_config.json алга. START = 'base' болгож болно.")
if not KIT:
    raise SystemExit("sat_desmos_kit.zip олдсонгүй (data/train.jsonl, technique талбартай).")
print("adapter:", ADAPTER, "· kit:", KIT)

kit_train, kit_val, kit_test = (read_jsonl(f"{KIT}/data/{n}.jsonl") for n in ("train", "val", "test"))
PROMPTS["desmos"] = kit_train[0]["messages"][0]["content"]
sys.path.insert(0, f"{KIT}/eval")
from scorer import score_one, summarize as desmos_summarize
print("desmos kit:", len(kit_train), "train ·", len(kit_val), "val ·", len(kit_test), "test")

# the side panel asks the same lessons in the app's format (choices as A) lines, LaTeX, the right value or letter)
res = subprocess.run([BUN, "repo/ai/v2/desmos-panel.ts", KIT, "panel"], capture_output=True, text=True)
if res.returncode != 0:
    print(res.stderr)
    raise SystemExit("ai/v2/desmos-panel.ts failed")
print(res.stdout.strip())
panel_train, panel_val, panel_test = (read_jsonl(f"panel/panel_{n}.jsonl") for n in ("train", "val", "test"))

In [ ]:
# ---- Load the model: first-training LoRA on its base model (or a fresh base) ----
from unsloth import FastLanguageModel
import torch

if START == "desmos":
    cfg = json.load(open(f"{ADAPTER}/adapter_config.json"))
    base_name = cfg["base_model_name_or_path"]
    targets = cfg["target_modules"]
    if isinstance(targets, str):   # e.g. "all-linear"
        targets = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"] \
            if targets == "all-linear" else [targets]
    lora = dict(r=cfg["r"], lora_alpha=cfg["lora_alpha"], target_modules=sorted(targets))
else:
    base_name = BASE_MODEL
    lora = dict(r=16, lora_alpha=16,
                target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
print("base:", base_name, "· LoRA:", lora)

model, tokenizer = FastLanguageModel.from_pretrained(model_name=base_name, max_seq_length=MAX_LEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, lora_dropout=0, bias="none", use_gradient_checkpointing="unsloth",
                                         random_state=42, **lora)

def lora_b_norm():
    return sum(p.detach().float().abs().sum().item() for n, p in model.named_parameters() if "lora_B" in n)

if START == "desmos":
    from peft import set_peft_model_state_dict
    path = f"{ADAPTER}/adapter_model.safetensors"
    if os.path.exists(path):
        from safetensors.torch import load_file
        weights = load_file(path)
    else:
        weights = torch.load(f"{ADAPTER}/adapter_model.bin", map_location="cpu")
    res = set_peft_model_state_dict(model, weights)
    bad = [k for k in getattr(res, "unexpected_keys", []) if "lora" in k]
    if bad:
        raise SystemExit(f"Adapter does not fit this model ({len(bad)} unexpected keys, e.g. {bad[0]})")
    # a fresh LoRA has lora_B = 0, so a non-zero norm proves the first training's weights are loaded
    assert lora_b_norm() > 0, "first-training weights did not load"
    print("first training loaded ✓ (lora_B norm", round(lora_b_norm(), 2), ")")

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

## Тест (сургалтын өмнө ба дараа)
Загварын хариуг код шалгана (`ai/v2/score.ts`, Desmos хичээлийг эхний kit-ийн `scorer.py`). Тест асуултууд сургалтад огт ороогүй.
- **chat**: хариу зөв % ↑ · hint-д хариу хэлсэн % ↓ · буруу хариуг "зөв" гэсэн % ↓ · монгол %
- **planner**: бүрэн зөв % (JSON, үе шат, сул сэдэв, завгүй өдөр, минутын нийлбэр, бүтэн тест) ↑
- **analyze**: бүрэн зөв % (бүх хэв маяг, тоо яг) ↑ · recall ↑
- **stepcheck**: эхний буруу алхмыг олсон % ↑
- **desmos**: эхний kit-ийн "full lesson passes" ↑
- **desmos_panel**: ижил шалгуур, апп-ын асуултын хэлбэрээр (`A) …` сонголт, LaTeX) ↑

In [ ]:
# ---- Batched generation + scoring ----
import random, math

def eval_rows():
    rows = []
    for skill in ("chat", "planner", "analyze", "stepcheck"):
        rows += [r for r in v2_test if r["skill"] == skill][: EVAL_N[skill]]
    by_tech = {}
    for r in kit_test:
        by_tech.setdefault(r["technique"], []).append(r)
    desmos = [lst[i] for i in range(10) for lst in by_tech.values() if i < len(lst)][: EVAL_N["desmos"]]
    rows += [{"skill": "desmos", "id": f"desmos-{i}", "messages": r["messages"][:2], "kit": r} for i, r in enumerate(desmos)]
    by_tech = {}
    for r in panel_test:
        by_tech.setdefault(r["technique"], []).append(r)
    panel = [lst[i] for i in range(10) for lst in by_tech.values() if i < len(lst)][: EVAL_N["desmos_panel"]]
    rows += [{"skill": "desmos_panel", "id": f"panel-{i}", "messages": r["messages"][:2], "kit": r} for i, r in enumerate(panel)]
    return rows

def tok_len(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

# answer length limit per skill, from the longest gold answers (+20%)
MAX_NEW = {}
for skill in ("chat", "planner", "analyze", "stepcheck"):
    golds = [r["messages"][-1]["content"] for r in v2_train if r["skill"] == skill][:400]
    MAX_NEW[skill] = min(1200, int(max(tok_len(g) for g in golds) * 1.2) + 32)
MAX_NEW["desmos"] = min(1500, int(max(tok_len(r["messages"][2]["content"]) for r in kit_test) * 1.2) + 32)
MAX_NEW["desmos_panel"] = MAX_NEW["desmos"]   # same gold lessons
print("max new tokens:", MAX_NEW)

def generate_batch(msgs_list, max_new):
    texts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs_list]
    enc = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    n = enc["input_ids"].shape[1]
    return [tokenizer.decode(o[n:], skip_special_tokens=True).strip() for o in out]

def run_model(rows, label):
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"   # batched generation needs left padding
    outs = {}
    for skill in dict.fromkeys(r["skill"] for r in rows):
        todo = sorted([r for r in rows if r["skill"] == skill], key=lambda r: len(r["messages"][1]["content"]))
        for i in range(0, len(todo), BATCH):
            part = todo[i : i + BATCH]
            for r, o in zip(part, generate_batch([r["messages"] for r in part], MAX_NEW[skill])):
                outs[r["id"]] = o
        print(f"{label}: {skill} done ({len(todo)})")
    return outs

def score(rows, outs, label):
    path = f"outputs_{label}.jsonl"
    with open(path, "w", encoding="utf-8") as fh:
        for r in rows:
            fh.write(json.dumps({"id": r["id"], "skill": r["skill"], "output": outs[r["id"]]}, ensure_ascii=False) + "\n")
    res = subprocess.run([BUN, "repo/ai/v2/score.ts", f"{V2}/test.jsonl", path], capture_output=True, text=True)
    if res.returncode != 0:
        print(res.stderr)
        raise SystemExit("scorer failed")
    summary = json.loads(res.stdout)
    for skill in ("desmos", "desmos_panel"):   # Desmos lessons: the v1 kit's scorer
        desmos = [r for r in rows if r["skill"] == skill]
        if not desmos:
            continue
        try:
            d = desmos_summarize([score_one(outs[r["id"]], r["kit"]) for r in desmos])
            summary[skill] = {"n": len(desmos), **(d if isinstance(d, dict) else {"result": str(d)})}
        except Exception as e:  # keep the run going if the kit's scorer API differs
            summary[skill] = {"n": len(desmos), "error": repr(e)}
    return summary

def evaluate(label):
    rows = eval_rows()
    outs = run_model(rows, label)
    summary = score(rows, outs, label)
    print(json.dumps(summary, ensure_ascii=False, indent=1))
    return {"model": label, "summary": summary, "outputs": outs, "rows": rows}

# self-test of the scorer on the gold answers (must be 100)
gold_rows = [r for r in v2_test if r["skill"] != "chat"]
with open("gold.jsonl", "w", encoding="utf-8") as fh:
    for r in gold_rows:
        fh.write(json.dumps({"id": r["id"], "output": r["check"]["gold"]}, ensure_ascii=False) + "\n")
print("scorer self-test:", subprocess.run([BUN, "repo/ai/v2/score.ts", f"{V2}/test.jsonl", "gold.jsonl"],
                                          capture_output=True, text=True).stdout)

In [ ]:
before = evaluate("before") if BEFORE_EVAL else None

## Сургалт
v2 өгөгдөл (chat, planner, analyze, stepcheck) + эхний kit-ийн Desmos хичээлүүдээс `CAP["desmos"]` + тэр хичээлүүдийг апп-ын хажуугийн самбарын хэлбэрээр асуусан `CAP["desmos_panel"]`-ийг холино. Зөвхөн Суугийн хариу дээр сурна (`train_on_responses_only`).

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
import inspect

random.seed(42)
def take(rows, n):
    rows = list(rows)
    random.shuffle(rows)
    return rows[:n]

fit = []
for skill in ("chat", "planner", "analyze", "stepcheck"):
    fit += take([r["messages"] for r in v2_train if r["skill"] == skill], CAP[skill])
fit += take([r["messages"] for r in kit_train], CAP["desmos"])
fit += take([r["messages"] for r in panel_train], CAP["desmos_panel"])
val = take([r["messages"] for r in v2_val], 160) + take([r["messages"] for r in kit_val], 40) \
    + take([r["messages"] for r in panel_val], 20)

def to_text(rows):
    texts = [tokenizer.apply_chat_template(m, tokenize=False) for m in rows]
    keep = [t for t in texts if tok_len(t) <= MAX_LEN]
    if len(keep) < len(texts):
        print(f"dropped {len(texts) - len(keep)} rows longer than {MAX_LEN} tokens")
    return Dataset.from_list([{"text": t} for t in keep])

random.shuffle(fit)
train_ds, val_ds = to_text(fit), to_text(val)
print("training on", len(train_ds), "rows · val", len(val_ds))

params = inspect.signature(SFTConfig.__init__).parameters
args = dict(dataset_text_field="text", per_device_train_batch_size=2, gradient_accumulation_steps=4,
            num_train_epochs=EPOCHS, learning_rate=LR, warmup_steps=10, lr_scheduler_type="linear",
            logging_steps=20, eval_steps=200, optim="adamw_8bit", weight_decay=0.01, seed=42,
            output_dir="outputs", report_to="none", packing=False)
args["max_seq_length" if "max_seq_length" in params else "max_length"] = MAX_LEN
args["eval_strategy" if "eval_strategy" in params else "evaluation_strategy"] = "steps"

FastLanguageModel.for_training(model)
tokenizer.padding_side = "right"
common = dict(model=model, train_dataset=train_ds, eval_dataset=val_ds, args=SFTConfig(**args))
try:
    trainer = SFTTrainer(processing_class=tokenizer, **common)
except TypeError:
    trainer = SFTTrainer(tokenizer=tokenizer, **common)
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")
stats = trainer.train()
print(stats)

In [ ]:
after = evaluate("after")

HEAD = [("chat", "answer_correct", "Чат: хариу зөв % ↑"), ("chat", "hint_leak", "Чат: hint-д хариу хэлсэн % ↓"),
        ("chat", "false_praise", "Чат: буруу хариуг зөв гэсэн % ↓"), ("stepcheck", "pass", "Буруу алхам олсон % ↑"),
        ("planner", "pass", "Төлөвлөгөө бүрэн зөв % ↑"), ("planner", "busy", "Завгүй өдрийг тооцсон % ↑"),
        ("analyze", "pass", "Хэв маяг бүрэн зөв % ↑"), ("analyze", "recall", "Хэв маяг олсон (recall) % ↑")]

def desmos_head(s, skill="desmos"):
    d = s.get(skill, {})
    for k, v in d.items():
        if isinstance(v, (int, float)) and ("full" in k or "pass" in k):
            return k, v
    return None, None

lines = ["# Suu v2 тайлан", "", f"Сургалт: {len(train_ds)} мөр · {EPOCHS} epoch · эхлэл: {START} ({base_name})", "",
         "| Хэмжүүр | Өмнө | Дараа |", "|---|---|---|"]
for skill, key, label in HEAD:
    b = before["summary"].get(skill, {}).get(key, "—") if before else "—"
    a = after["summary"].get(skill, {}).get(key, "—")
    lines.append(f"| {label} | {b} | {a} |")
dk, dv = desmos_head(after["summary"])
if dk:
    lines.append(f"| Desmos хичээл ({dk}) % ↑ | {desmos_head(before['summary'])[1] if before else '—'} | {dv} |")
pk, pv = desmos_head(after["summary"], "desmos_panel")
if pk:
    pb = desmos_head(before["summary"], "desmos_panel")[1] if before else "—"
    lines.append(f"| Desmos самбар, апп-ын асуулт ({pk}) % ↑ | {pb} | {pv} |")
lines += ["", "## Бүх тоо", "", "```json", json.dumps({"before": before and before["summary"], "after": after["summary"]},
                                                      ensure_ascii=False, indent=1), "```", "", "## Жишээ хариу (дараа)", ""]
for r in after["rows"]:
    if r["skill"] in ("planner", "stepcheck", "analyze") and r["id"].endswith(("-0", "-1")):
        lines += [f"### {r['skill']} · {r['id']}", "", "```", after["outputs"][r["id"]][:1500], "```", ""]
report = "\n".join(lines)
open("eval_report_v2.md", "w", encoding="utf-8").write(report)
json.dump({"before": before and before["summary"], "after": after["summary"]}, open("results_v2.json", "w"),
          ensure_ascii=False, indent=1)

import matplotlib.pyplot as plt
labels = [h[2] for h in HEAD]
a_vals = [after["summary"].get(s, {}).get(k, 0) for s, k, _ in HEAD]
b_vals = [before["summary"].get(s, {}).get(k, 0) for s, k, _ in HEAD] if before else [0] * len(HEAD)
fig, ax = plt.subplots(figsize=(10, 5))
y = range(len(labels))
ax.barh([i + 0.2 for i in y], b_vals, height=0.4, label="Өмнө (эхний сургалт)", color="#c9c9c9")
ax.barh([i - 0.2 for i in y], a_vals, height=0.4, label="Дараа (Суу v2)", color="#2e7d5b")
ax.set_yticks(list(y)); ax.set_yticklabels(labels); ax.invert_yaxis(); ax.set_xlim(0, 100)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.07), ncol=2, frameon=False)
ax.set_title("Суу v2: сургалтын өмнө ба дараа (тест асуулт сургалтад ороогүй)")
plt.tight_layout(); plt.savefig("results_v2_chart.png", dpi=150)
from IPython.display import Markdown, display
display(Markdown(report))

## Туршиж үзэх

In [ ]:
FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"
demo_plan = {"days": ["Да", "Мя", "Лх", "Пү", "Ба", "Бя", "Ня"], "daysLeft": 40, "current": 560, "target": 680,
             "hoursPerDay": 1.5,
             "topics": [{"id": 4, "name": "Шугаман тэгшитгэлийн систем", "acc": 45, "n": 20, "sec": 110, "trend": -5},
                        {"id": 9, "name": "Хувь", "acc": 70, "n": 15, "sec": 80, "trend": 0},
                        {"id": 17, "name": "Квадрат тэгшитгэл", "acc": 52, "n": 18, "sec": 140, "trend": 0}],
             "habits": {"rushed": 4, "slow": 1, "desmos": 2},
             "note": "Мягмар гарагт дугуйлантай. Бямба гарагт 3 цаг гаргаж чадна."}
demo_step = ("Solve for $x$: $4(x - 3) = 2x + 10$\n\nМиний бодолт:\n1) $4x - 3 = 2x + 10$\n2) $2x = 13$\n"
             "3) $x = 6.5$\n\nАлдаа маань хаана байна?")
demos = {
    "planner": generate_batch([[{"role": "system", "content": PROMPTS["planner"]},
                                {"role": "user", "content": json.dumps(demo_plan, ensure_ascii=False)}]], MAX_NEW["planner"])[0],
    "stepcheck": generate_batch([[{"role": "system", "content": PROMPTS["chat"]},
                                  {"role": "user", "content": demo_step}]], MAX_NEW["stepcheck"])[0],
    "chat_latin": generate_batch([[{"role": "system", "content": PROMPTS["chat"]},
                                   {"role": "user", "content": "If $2x+3=11$, what is $x$?\n\nhint ogooch"}]], 200)[0],
}
for k, v in demos.items():
    print(f"--- {k}\n{v}\n")
json.dump(demos, open("demo_v2.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

## GGUF → Ollama

`suu_v2.zip`-ийг laptop дээрээ задлаад:

```bash
cd suu-v2
ollama create suu -f Modelfile
ollama run suu "Сайн уу, Суу!"
```

`prompts.json` дотор чадвар бүрийн system prompt бий — апп `planner`, `analyze`, `desmos` чадварыг тэр prompt-оор дуудна. `suu_v2_lora.zip`-ийг хадгал (дараагийн сургалт үүнээс эхэлнэ).

In [ ]:
import shutil
model.save_pretrained("suu-v2-lora"); tokenizer.save_pretrained("suu-v2-lora")
model.save_pretrained_gguf("suu-v2-gguf", tokenizer, quantization_method="q4_k_m")

os.makedirs("suu-v2", exist_ok=True)
gguf = sorted(glob.glob("suu-v2-gguf*/**/*.gguf", recursive=True) + glob.glob("*.gguf"), key=os.path.getsize)[-1]
shutil.copy(gguf, "suu-v2/suu.gguf")
sys_prompt = PROMPTS["chat"].replace('"', '\\"')
modelfile = f'''FROM ./suu.gguf
TEMPLATE """{{{{- if .System }}}}<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{ end }}}}{{{{- range .Messages }}}}<|im_start|>{{{{ .Role }}}}
{{{{ .Content }}}}<|im_end|>
{{{{ end }}}}<|im_start|>assistant
"""
SYSTEM "{sys_prompt}"
PARAMETER temperature 0.3
PARAMETER num_ctx 4096
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|im_start|>"
'''
open("suu-v2/Modelfile", "w", encoding="utf-8").write(modelfile)
json.dump(PROMPTS, open("suu-v2/prompts.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
for f in ("eval_report_v2.md", "results_v2.json", "results_v2_chart.png", "demo_v2.json"):
    if os.path.exists(f):
        shutil.copy(f, f"suu-v2/{f}")
!zip -r -q suu_v2.zip suu-v2
!zip -r -q suu_v2_lora.zip suu-v2-lora
!ls -lh suu-v2 suu_v2.zip suu_v2_lora.zip
from google.colab import files
files.download("suu_v2_lora.zip")   # small — download first
files.download("suu_v2.zip")